# 프로젝트 목표

대규모 언어 모델(LLM)의 핵심 구조인 **Transformer**는 공개된 연구를 통해 동작 원리가 널리 알려져 있습니다. 하지만 실제 LLM은 수십억~수조 개의 파라미터를 학습하며, 동일한 질문에 대해서도 내부적으로 어떤 과정을 거쳐 최종 답변을 생성하는지 정확하게 예측하기는 매우 어렵습니다.

이러한 특성 때문에 최근에는 단순히 모델의 크기를 키우는 것뿐만 아니라, **LLM이 더 정확하고 일관된 답변을 생성할 수 있도록 다양한 추론(Reasoning) 및 성능 개선 기법**이 활발히 연구되고 있습니다. 대표적으로 **Chain of Thought(CoT)**, **Self-Consistency(SC)**, **ReAct**, **RAG**, **Reflection** 등의 기법이 실제 서비스에서 활용되고 있습니다.

이번 실습에서는 이러한 고도화 기법을 적용하기에 앞서, **LLM의 가장 기본이 되는 Transformer의 핵심 구조를 직접 구현**하여 LLM이 입력 문장을 어떻게 처리하고 다음 토큰을 예측하는지 이해하는 것을 목표로 합니다.

## 구현 내용

1. **Token Embedding**
2. **Positional Encoding**
3. **Self Attention**
4. **Multi Head Attention**
5. **Transformer Block**
6. **Next Token Prediction**

이번 실습은 **LLM의 기본 동작 원리를 이해하기 위한 Transformer 기반 구현**이며, 이후 학습에서는 이 구조를 기반으로 **CoT, Self-Consistency(SC), ReAct**와 같은 추론 기법이 어떻게 LLM의 응답 품질을 향상시키는지 단계적으로 학습하게 됩니다.

In [48]:
# ==========================================================
# TO DO 0. 환경 설정 및 패키지 설치
# ==========================================================
#
# Transformer 구현에 필요한 PyTorch를 설치합니다.
#
# Colab 환경에서는 실행 가능
#
# VS Code에서는 Terminal에서 설치 가능
#
# ==========================================================


# Colab 실행 시

!pip install torch -q



# ==========================================================
# 설치 확인
# ==========================================================

import torch


print("PyTorch Version:")
print(torch.__version__)



# GPU 사용 가능 여부 확인

print("\nDevice Check")

if torch.cuda.is_available():

    print("GPU 사용 가능")
    print(torch.cuda.get_device_name(0))

else:

    print("CPU 사용")

PyTorch Version:
2.11.0+cpu

Device Check
CPU 사용


In [49]:
# ==========================================================
# 프로젝트명
# Mini LLM Transformer Builder
# ==========================================================
#
# [프로젝트 시나리오]
#
# AI Research Lab에서는
# 작은 규모의 언어 모델(Mini LLM)을 개발하고 있습니다.
#
# AI 엔지니어인 여러분은
# Transformer의 핵심 구조를 직접 구현하고,
# 입력 데이터에 대한 다음 Token 예측 모델을
# 만들어야 합니다.
#
#
# 이번 실습에서는
#
# ① Token Embedding
# ② Positional Encoding
# ③ Self Attention
# ④ Multi Head Attention
# ⑤ Transformer Block
# ⑥ Next Token Prediction
#
# 을 직접 구현합니다.
#
#
# 현재 실습은 Transformer 구조의 Forward 동작을 확인하는 단계입니다.
# 실제 LLM처럼 의미 있는 Token을 생성하려면 학습 데이터, Loss Function,
# Backpropagation, Training Loop 과정이 추가되어야 합니다.
# 따라서 검증 기준은
# Input shape:
# torch.Size([1,5])
#
# Output shape:
# torch.Size([1,5,50])

# Predicted Token:
# 0~49 사이 숫자이면 성공입니다.
# ==========================================================



# ==========================================================
# TO DO 1. 라이브러리 Import
# ==========================================================
#
# 아래 라이브러리를 import 하시오.
#
# 필요 라이브러리
#
# • torch
# • torch.nn
# • torch.nn.functional
# • math
#
#
# Hint
#
# import torch
#
# ==========================================================


import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ==========================================================
# TO DO 2. Positional Encoding 클래스 작성
# ==========================================================
#
# Transformer는 순서 정보를 자동으로 기억하지 못합니다.
#
# 따라서 입력 Token에 위치 정보를 추가해야 합니다.
#
#
# 클래스명
#
# PositionalEncoding
#
#
# 조건
#
# ① nn.Module 상속
#
# ② position 생성
#
# ③ sin, cos 위치 정보 계산
#
# ④ 입력 embedding과 위치 정보 더하기
#
#
# Hint
#
# torch.sin()
#
# torch.cos()
#
# ==========================================================


class PositionalEncoding(nn.Module):

    # 코드 작성
    def __init__(self, dim, max_len=100):
      super().__init__()

      # 위치벡터
      position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

      # 위치에 곱해줄 값
      div_term = torch.exp(torch.arange(0, dim, 2) * -(math.log(10000) / dim))

      pe = torch.zeros(max_len, dim)
      #  포지셔널 인코딩 마지막 차원이 짝수 (슬라이싱 0::2 -> 0부터 시작해서 스텝 2씩이니까 짝수)
      pe[:, 0::2] = torch.sin(position * div_term)
      #  포지셔널 인코딩 마지막 차원이 홀수 (슬라이싱 1::2 -> 1부터 시작해서 스텝 2이니까 홀수)
      pe[:, 1::2] = torch.cos(position * div_term)

      #  차원 추가 - 임베딩 결과값이랑 더해야하니까 shape 맞추기
      pe = pe.unsqueeze(0)

      #  pe 벡터를 buffer 로 register ->> 학습할 때 update 되지 않도록 고정 (데이터 전송하는동안 일시적으로 데이터를 보관하는 메모리)
      self.register_buffer('pe', pe)


    def forward(self, x):
      # B, T = x.shape -> x가 2차원 일 때만 동작
      seq_length = x.size(1)

      # 입력 시퀀스의 길이에 맞춰 슬라이싱
      return self.pe[:, :seq_length, :]



# ==========================================================
# TO DO 3. Scaled Dot Product Attention 구현
# ==========================================================
#
# Attention의 핵심 연산을 구현하시오.
#
#
# 함수명
#
# scaled_dot_product_attention()
#
#
# 입력
#
# Q
# K
# V
#
#
# 처리 과정
#
# ① Q와 K 내적 계산
#
# ② sqrt(d_k)로 나누기
#
# ③ Softmax 적용
#
# ④ Value와 곱하기
#
#
# 반환
#
# output
# attention weight
#
#
# Hint
#
# torch.matmul()
#
# F.softmax()
#
# ==========================================================


def scaled_dot_product_attention(Q, K, V):

    # 코드 작성

    d_k = Q.size(-1)

    # Q와 K 내적 계산, sqrt(d_k)로 나누기
    # Q K 곱해서 단어끼리 얼마나 관련 있는지 점수 매김
    # math.sqrt(d_k) 로 나누는 이유가 뭘까.. -> 차원이 커지면 내적값도 같이 커져서 softmax 가 한쪽으로 쏠리는데, 그 문제를 방지
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Softmax 적용 - 점수를 모두 더하면 1이 되도록 바꿔줌
    attention_weight = F.softmax(scores, dim=-1)

    # Value와 곱하기
    output = torch.matmul(attention_weight, V)

    return output, attention_weight





# ==========================================================
# TO DO 4. Multi Head Attention 구현
# ==========================================================
#
# 여러 개의 Attention Head를 사용하는
# Multi Head Attention을 구현하시오.
#
#
# 클래스명
#
# MultiHeadAttention
#
#
# 조건
#
# ① Query Linear Layer 생성
#
# ② Key Linear Layer 생성
#
# ③ Value Linear Layer 생성
#
# ④ Head 분리
#
# ⑤ Attention 수행
#
# ⑥ Head 결합
#
#
# Hint
#
# nn.Linear()
#
# view()
#
# transpose()
#
#
# ==========================================================


class MultiHeadAttention(nn.Module):

    # 코드 작성

    def __init__(self, d_model, num_heads):
      super().__init__()
      self.d_model = d_model
      self.num_heads = num_heads
      self.head_dim = d_model // num_heads  # head 개수만큼 쪼개 쓸 거라 나눠줌

      self.query_linear = nn.Linear(d_model, d_model)
      self.key_linear = nn.Linear(d_model, d_model)
      self.value_linear = nn.Linear(d_model, d_model)

      # head 여러 개 결과 합친 다음에 한 번 더 섞어주는 층
      self.w_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
      batch_size = Q.size(0)

      # ① Query Linear Layer 생성
      Q = self.query_linear(Q)

      # ② Key Linear Layer 생성
      K = self.key_linear(K)

      # ③ Value Linear Layer 생성
      V = self.value_linear(V)


      # ④ Head 분리
      Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
      K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
      V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

      # ⑤ Attention 수행
      attention_output, attention_weight = scaled_dot_product_attention(Q, K, V)

      # ⑥ Head 결합
      # transform 다시 원복, transpose하면 메모리 이어져있지않아서 contiguous
      output = attention_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

      return self.w_o(output)



# ==========================================================
# TO DO 5. Transformer Block 구현
# ==========================================================
#
# Transformer의 하나의 Block을 작성하시오.
#
#
# 포함 요소
#
# ① Multi Head Attention
#
# ② Feed Forward Network
#
# ③ Residual Connection
#
# ④ Layer Normalization
#
#
# 클래스명
#
# TransformerBlock
#
#
# Hint
#
# nn.Sequential()
#
# nn.LayerNorm()
#
# ==========================================================


class TransformerBlock(nn.Module):

    # 코드 작성

    def __init__(self, dim, num_heads):
      super().__init__()

      self.attention = MultiHeadAttention(dim, num_heads)

      self.norm1 = nn.LayerNorm(dim)
      self.norm2 = nn.LayerNorm(dim)

      # attention 은 값을 섞음, 차원 4배로 확 늘렸다가 다시 원래 차원으로
      self.feed_forward = nn.Sequential(
          nn.Linear(dim, dim * 4),
          nn.ReLU(),
          nn.Linear(dim * 4, dim)
      )

    def forward(self, x):
      # ① Multi Head Attention
      attention_output = self.attention(x, x, x)
      # 잔차연결 (Residual)
      x = x + attention_output
      # 더한 값 분포 정리 (Normalization) -> 안하면 층 쌓일 수록 학습 불안정해짐
      x = self.norm1(x)

      # ② Feed Forward Network
      feed_forward_output = self.feed_forward(x)
      x = x + feed_forward_output
      x = self.norm2(x)

      return x


# ==========================================================
# TO DO 6. Mini LLM 모델 작성
# ==========================================================
#
# Transformer를 이용한
# Mini LLM 모델을 작성하시오.
#
#
# 클래스명
#
# MiniLLM
#
#
# 구성 요소
#
# ① Token Embedding
#
# ② Positional Encoding
#
# ③ Transformer Layer
#
# ④ Output Linear Layer
#
#
# 출력:
#
# Vocabulary 크기만큼의 예측값
#
#
# Hint
#
# nn.Embedding()
#
# nn.ModuleList()
#
# nn.Linear()
#
# ==========================================================


class MiniLLM(nn.Module):

    # 코드 작성
    def __init__(self, vocab_size, dim=64, num_heads=4, num_layers=3):
      super().__init__()
      self.embedding = nn.Embedding(vocab_size, dim)
      self.positional_encoding = PositionalEncoding(dim)

      # TransformerBlock 을 num_layers 개수만큼 쌓음
      self.blocks = nn.ModuleList([
          TransformerBlock(dim, num_heads) for _ in range(num_layers)
      ])

      # 다음 토큰이 뭘지 vocab 전체에 대한 확률 볼 수 있음
      self.lm_head = nn.Linear(dim, vocab_size)



    def forward(self, x):
      # ① Token Embedding
      token = self.embedding(x)

      # ② Positional Encoding
      position = self.positional_encoding(x)

      #  의미정보 + 위치 정보 = >진짜 입력
      x = token + position

      # ③ Transformer Layer
      for block in self.blocks:
        x = block(x)

      return self.lm_head(x)



# ==========================================================
# TO DO 7. 테스트 데이터 생성
# ==========================================================
#
# 모델 테스트를 위한 입력 데이터를 생성하시오.
#
#
# 조건
#
# vocabulary size
#
# 50
#
#
# batch size
#
# 1
#
#
# sequence length
#
# 5
#
#
# Hint
#
# torch.randint()
#
# ==========================================================


vocab_size = 50


x = torch.randint(0, vocab_size, (1, 5))





# ==========================================================
# TO DO 8. 모델 생성 및 Forward 실행
# ==========================================================
#
# MiniLLM 모델을 생성하고
# 입력 데이터를 전달하시오.
#
#
# 조건
#
# model 생성
#
# output 생성
#
#
# Hint
#
# output = model(x)
#
# ==========================================================


model = MiniLLM(vocab_size)


output = model(x)





# ==========================================================
# TO DO 9. 다음 Token 예측
# ==========================================================
#
# 마지막 위치의 출력값을 이용하여
# 다음 Token을 예측하시오.
#
#
# 과정
#
# ① 마지막 Token 출력 선택
#
# ② Softmax 적용
#
# ③ 가장 높은 확률 Token 선택
#
#
# Hint
#
# torch.argmax()
#
# F.softmax()
#
# ==========================================================


next_token_logits = output[0, -1]


prediction = torch.argmax(F.softmax(next_token_logits, dim=-1)).item()





# ==========================================================
# TO DO 10. 결과 출력
# ==========================================================
#
# 다음 내용을 출력하시오.
#
#
# Input Shape
#
# Output Shape
#
# Predicted Token
#
#
# ==========================================================


print("Input shape:")
print(x.shape)
print()
print("Output shape:")
print(output.shape)
print()
print("Predicted Token:")
print(prediction)



# ==========================================================
# 실행 결과 예시
# ==========================================================
#
# Input shape:
# torch.Size([1,5])
#
#
# Output shape:
# torch.Size([1,5,50])
#
#
# Predicted Token:
# 숫자
#
# ==========================================================






# ==========================================================
# 추가 학습
# ==========================================================
#
# 현재 구현한 모델은
# Transformer 기본 구조 학습용입니다.
#
#
# 실제 GPT 모델과 차이점
#
# ① Causal Mask 필요
#
# ② Tokenizer 필요
#
# ③ Loss 계산 필요
#
# ④ Backpropagation 필요
#
# ⑤ Training Loop 필요
#
# ==========================================================

Input shape:
torch.Size([1, 5])

Output shape:
torch.Size([1, 5, 50])

Predicted Token:
0


In [50]:
# ==========================================================
# VS Code 실행 환경 안내
# ==========================================================
#
# 1. 가상환경 생성
#
# python -m venv venv
#
#
# 2. 가상환경 활성화
#
# Windows:
#
# venv\Scripts\activate
#
#
# 3. PyTorch 설치
#
# pip install torch
#
#
# 4. 실행
#
# python mini_llm.py
#
# ==========================================================